<a href="https://colab.research.google.com/github/balajiduddukuri/Langchain_practice/blob/Ultimate-Content-Repurposer/Langchain_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
#practice programs

In [1]:

# ===============================
# 1) Install dependencies
# ===============================
!pip install -q langchain langchain-google-genai google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00


In [2]:

# Optional utilities if you plan to extend to RAG later:
# !pip install -q chromadb faiss-cpu pypdf tiktoken docarray

# ===============================
# 2) Imports & API key setup
# ===============================
import os
from google.colab import userdata

# LangChain core + Google Gemini Chat wrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

# Load and assert API key (store it in Colab with key name: "google_api_key")
os.environ["GOOGLE_API_KEY"] = userdata.get("google_api_key")
assert os.environ.get("GOOGLE_API_KEY"), (
    "Missing GOOGLE_API_KEY. In Colab, run: "
    "from google.colab import userdata; userdata.set('google_api_key', 'YOUR_KEY')"
)
print("Google API key is loaded successfully.")

Google API key is loaded successfully.


In [ ]:

# ===============================
# 3) Helper: build Gemini LLM
# ===============================
def get_llm(model: str = "gemini-1.5-flash", temperature: float = 0):
    """
    Returns a LangChain LLM wrapper for Google Gemini chat models.
    Common models:
      - gemini-1.5-flash : fast, cheaper, good for most chat tasks
      - gemini-1.5-pro   : higher quality, reasoning
    """
    return ChatGoogleGenerativeAI(model=model, temperature=temperature)

In [10]:
import os
import requests

API_KEY = os.getenv("google_api_key")
if not API_KEY:
    raise SystemExit("Set GEMINI_API_KEY in your environment.")

# Primary endpoint (v1). Some accounts/regions might still expose models under v1beta.
BASES = [
    "https://generativelanguage.googleapis.com/v1/models",
    "https://generativelanguage.googleapis.com/v1beta/models",  # fallback
]

def list_gemini_models() -> list[str]:
    for base in BASES:
        try:
            r = requests.get(base, params={"key": API_KEY}, timeout=20)
            r.raise_for_status()
            data = r.json()
            models = data.get("models", [])
            names = []
            for m in models:
                # Example: "name": "models/gemini-1.5-flash"
                n = m.get("name", "")
                if n.startswith("models/"):
                    n = n.split("/", 1)[1]
                if n:
                    names.append(n)
            if names:
                return sorted(set(names))
        except requests.HTTPError as e:
            # try the next base; if none works, rethrow on last iteration
            if base is BASES[-1]:
                raise
            continue
    return []



In [11]:
#if __name__ == "__main__":
names = list_gemini_models()
print("\n=== Gemini models (live) ===")
if not names:
   print("(No models returned. Check your API key or permissions.)")
for n in names:
   print(" -", n)

HTTPError: 400 Client Error: Bad Request for url: https://generativelanguage.googleapis.com/v1beta/models?key=google_api_key

In [ ]:

# ===============================
# 4) Run the same query with two Gemini models
# ===============================
query = "Explain the uses of LangChain Framework in bullet points"

for model in ["gemini-1.5-flash", "gemini-1.5-pro"]:
    llm = get_llm(model=model, temperature=0)
    response = llm.invoke([HumanMessage(content=query)])
    print(f"\n--- {model} ---\n{response.content}")

In [ ]:
# ===============================
# 5) PromptTemplate demo (unchanged)
# ===============================
template = """
You are an expert AI tutor.

Explain {topic} in simple terms.
Give examples.
Audience: {audience}
"""

prompt = PromptTemplate(
    input_variables=["topic", "audience"],
    template=template
)

formatted_prompt = prompt.format(
    topic="GAN",
    audience="Beginner AI Students"
)

print("\n--- Formatted Prompt ---\n", formatted_prompt)

llm = get_llm("gemini-1.5-flash", temperature=0)
resp = llm.invoke(formatted_prompt)  # You can pass a plain string to invoke
print("\n--- Model Output ---\n", resp.content)

In [ ]:
# pip install markdown
from markdown import markdown

text = """YOUR_LONG_TEXT_HERE"""

html = markdown(resp, extensions=["extra"])  # 'extra' handles tables, fenced code, etc.
print(html)